In [ ]:
# 1. Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 2. Load Dataset
df = pd.read_csv("bank-additional-full.csv", sep=";")  # Update path and separator if needed


df['y'] = df['y'].map({'yes': 1, 'no': 0})

# 4. Separate features and target
X = df.drop('y', axis=1)
y = df['y']

# 5. Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 6. Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# 7. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)],
    remainder='passthrough'
)

# 8. Define model pipelines
def build_pipeline(model):
    return Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])

# 9. Define parameter grids
param_grids = {
    'Logistic Regression': {
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__solver': ['liblinear', 'lbfgs']
    },
    'Decision Tree': {
        'classifier__max_depth': [None, 5, 10, 20],
        'classifier__min_samples_split': [2, 5],
        'classifier__criterion': ['gini', 'entropy']
    },
    'Random Forest': {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2]
    },
    'Gradient Boosting': {
        'classifier__n_estimators': [100, 150],
        'classifier__learning_rate': [0.01, 0.1, 0.2],
        'classifier__max_depth': [3, 5, 10]
    }
}

# 10. Run GridSearchCV for all models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\nRunning GridSearchCV for {name}...")
    pipeline = build_pipeline(model)
    grid = GridSearchCV(pipeline, param_grids[name], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    print(f"Best Parameters for {name}: {grid.best_params_}")
    print(f"Best Cross-Validation Accuracy for {name}: {grid.best_score_:.4f}")
    
    y_pred = grid.predict(X_test)
    print(f"\n{name} Classification Report on Test Set:")
    print(classification_report(y_test, y_pred))
    
    results[name] = {
        'model': grid,
        'accuracy': grid.best_score_
    }

# 11. Optional: Show comparison
print("\n\n📊 Model Accuracy Comparison (CV Score):")
for name, res in results.items():
    print(f"{name}: {res['accuracy']:.4f}")



Running GridSearchCV for Logistic Regression...


/opt/conda/envs/anaconda-panel-2023.05-py310/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/conda/envs/anaconda-panel-2023.05-py310/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/

Best Parameters for Logistic Regression: {'classifier__C': 10, 'classifier__solver': 'liblinear'}
Best Cross-Validation Accuracy for Logistic Regression: 0.9098

Logistic Regression Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.93      0.98      0.95      7310
           1       0.71      0.39      0.51       928

    accuracy                           0.91      8238
   macro avg       0.82      0.69      0.73      8238
weighted avg       0.90      0.91      0.90      8238


Running GridSearchCV for Decision Tree...
Best Parameters for Decision Tree: {'classifier__criterion': 'gini', 'classifier__max_depth': 5, 'classifier__min_samples_split': 2}
Best Cross-Validation Accuracy for Decision Tree: 0.9120

Decision Tree Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95      7310
           1       0.68      0.53      0.59       928

    accur